# 🚁 Mission Complète Tello EDU : Exploration + SLAM Visuel + Export

Ce notebook exécute une mission autonome complète sur le drone réel tout en générant une carte SLAM en temps réel.

**Fonctionnalités activées :**
1.  **Contrôle de mission** : Gestion automatique des waypoints (`ExplorationMission`).
2.  **SLAM Visuel** : Cartographie de l'environnement via la caméra (`VisualSLAM`).
3.  **Visualisation** : Affichage de la grille d'occupation.
4.  **Export** : Sauvegarde des rapports et des cartes.

In [ ]:
%matplotlib inline
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

# Import des modules du projet
from exploration import ExplorationMission, MissionConfig, MissionStatus
from visual_slam import VisualSLAM
from vision import VideoStream

print("Modules chargés.")

### 1. Configuration et Initialisation
Nous configurons une petite zone d'exploration (ex: 200x200 cm) pour le test réel.

In [ ]:
# 1. Configuration de la mission
config = MissionConfig(
    area_width=200,          # Zone de 2m x 2m
    area_height=200,
    exploration_altitude=100, # Altitude 1m
    step_size=50,            # Pas de 50cm
    pattern="snake",
    enable_avoidance=True    # Activer l'évitement d'obstacles
)

# 2. Création de la mission (Mode Réel)
print("Initialisation de la mission...")
mission = ExplorationMission(config, simulation_mode=False)

# 3. Préparation (Connexion au drone)
if mission.prepare_mission():
    print("✅ Mission prête et drone connecté.")
    
    # Récupération de l'objet drone pour le SLAM
    real_drone = mission.controller.drone
    
    # 4. Initialisation du SLAM avec le drone connecté
    # Nous créons un flux vidéo lié au drone réel
    video_stream = VideoStream(drone=real_drone, simulation_mode=False)
    slam = VisualSLAM(video_stream=video_stream, simulation_mode=False)
    
else:
    print("❌ Erreur lors de la préparation (Vérifiez le WiFi).")

### 2. Exécution de la Mission
Le bloc suivant lance le SLAM (thread de fond) puis la mission (contrôle du drone). Une boucle surveille la progression et affiche la carte SLAM en temps réel.

In [ ]:
try:
    print("Démarrage du SLAM...")
    slam.start()
    # Laisser un peu de temps pour l'initialisation vidéo
    time.sleep(2)
    
    print("Démarrage de la Mission d'Exploration...")
    success = mission.start_exploration()
    
    if success:
        # Boucle de surveillance
        while mission.status == MissionStatus.IN_PROGRESS:
            # Récupérer les infos
            progress = mission.planner.get_progress()
            pos = mission.controller.position
            grid = slam.occupancy_grid
            
            # Visualisation graphique (SLAM Map)
            clear_output(wait=True)
            plt.figure(figsize=(10, 8))
            
            # Affichage de la grille (-1: inconnu, 0: libre, 100: occupé)
            # On utilise une colormap adaptée
            plt.imshow(grid, cmap='magma', vmin=-1, vmax=100, origin='lower')
            plt.colorbar(label='Probabilité d\'occupation')
            plt.title(f"Exploration : {progress:.1f}% | Pos: ({pos.x:.0f}, {pos.y:.0f})")
            plt.xlabel("X (Cellules)")
            plt.ylabel("Y (Cellules)")
            plt.show()
            
            time.sleep(0.5)
            
        print("Mission terminée !")
        
    else:
        print("Impossible de démarrer l'exploration.")

except KeyboardInterrupt:
    print("⚠️ Interruption manuelle !")
    mission.emergency_stop()
except Exception as e:
    print(f"❌ Erreur : {e}")
finally:
    print("Arrêt des systèmes...")
    mission.stop_exploration()
    slam.stop()

### 3. Exportation des Données
Sauvegarde des résultats de la mission (trajet, obstacles) et de la carte SLAM.

In [ ]:
import os

# Création du dossier de résultats
output_dir = "resultats_mission_reelle"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

timestamp = time.strftime("%Y%m%d-%H%M%S")
base_path = os.path.join(output_dir, f"mission_{timestamp}")

print(f"Exportation des données vers {base_path}...")

# 1. Export du rapport de mission (JSON, CSV)
mission.export_results(base_path)

# 2. Export de la carte SLAM (Image, Numpy, JSON)
slam.export_map(base_path)

print("✅ Export terminé. Fichiers générés :")
# Lister les fichiers créés (commande système compatible Linux/Mac/Windows via Python)
for f in os.listdir(output_dir):
    if f.startswith(f"mission_{timestamp}"):
        print(f" - {f}")